# StravaGANte Syntetic Data Retriever
The dataset is collected using OpenRouteService, an open source project which exposes free APIs.

In [22]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from serpapi import GoogleSearch
import csv
import sys
from time import sleep

IMDB 250 top films to scrape locations

In [ ]:
def print_progress_bar(iteration, total, length=50):
    percent = ("{0:.1f}").format(100 * (iteration / float(total)))
    filled_length = int(length * iteration // total)
    bar = '█' * filled_length + '-' * (length - filled_length)
    sys.stdout.write(f'\rProgress: |{bar}| {percent}%\n')
    sys.stdout.flush()

latlong_file = '../Data/latlong_movies.csv'
df_movies = pd.read_csv('../Data/IMDB_Top_250_Movies.csv', usecols=[1])
movie_names = df_movies['name'].tolist()
total_rows = len(movie_names)

movie_locations = {}

if os.path.isfile(latlong_file):
    existing_data = pd.read_csv(latlong_file)
    existing_data.dropna(subset=['name', 'latlong_url'], inplace=True)
    existing_data.to_csv(latlong_file, index=False)
    existing_movies = existing_data['name'].tolist()
    completed_rows = existing_data['latlong_url'].notna().sum()
else:
    existing_movies = []
    with open(latlong_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['name', 'latlong_url'])

load_dotenv()
# serpapi_api_key = os.getenv("serpapi_token")
serpapi_api_key = os.getenv("serpapi_token_secondary")
print("Token ok.") if serpapi_api_key else print("Token not found.")

for movie in movie_names:
    if movie not in existing_movies:
        print(f"{movie}...")
        query = f'site:latlong.net/location/ {movie}'
        try:
            search = GoogleSearch({
                "q": query,
                "location": "United States",
                "api_key": serpapi_api_key
            })
            results = search.get_dict()
            href = results['organic_results'][0]['link'] if 'organic_results' in results and results['organic_results'] else None
        except Exception as e:
            print(f"Error occurred: {e}")
            href = None
        
        if href:
            with open(latlong_file, mode='a', newline='') as file:
                writer = csv.writer(file)
                writer.writerow([movie, href])
            print(f"Found.")
            completed_rows += 1
        else:
            print(f"No results found.")
        
    print_progress_bar(completed_rows, total_rows)


Scrape from latlong.net every location.

In [30]:
import requests
from bs4 import BeautifulSoup

latlong_movies_copy_file = '../Data/latlong_movies.csv'
output_file = '../Data/latlong_movies_coordinates.csv'

# Read the CSV file
df_latlong = pd.read_csv(latlong_movies_copy_file)

# Create a list to store the results
results = []

# Iterate over each row in the DataFrame
for index, row in df_latlong.iterrows():
    url = row['latlong_url']
    movie_name = row['name']
    
    # Make a request to the URL
    response = requests.get(url)
    
    if response.status_code == 200:
        # Parse the HTML content
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the table in the page
        table = soup.find('table')
        
        if table:
            rows = table.find_all('tr')
            for r in rows[1:]:
                lat = r.find_all('td')[1].text.strip()
                lon = r.find_all('td')[2].text.strip()
                location_name = r.find_all('td')[0].text.strip()
            
                # Append the result to the list
                results.append([movie_name, location_name, lat, lon])
        else:
            print(f"No table found for {movie_name}")
    else:
        print(f"Failed to retrieve {url}")

# Create a DataFrame from the results
df_results = pd.DataFrame(results, columns=['name', 'location_name', 'latitude', 'longitude'])

# Save the results to a new CSV file
df_results.to_csv(output_file, index=False)

POST Request to OpenRouteService

In [ ]:
import random
from geopy.distance import geodesic
from datetime import datetime, timedelta

# Access Limits: (daily / per minute)
# Directions* (2.000 / 40) 
# --> 1 REQUEST EVERY 43.2 SECONDS!

profiles = [
    'driving-car',
    'driving-hgv',
    'cycling-regular',
    'cycling-road',
    'cycling-mountain',
    'cycling-electric'
]
output_filedir = '../Data/Syntetic/'
os.makedirs(output_filedir, exist_ok=True)
locs_file = '../Data/latlong_movies_coordinates.csv'

load_dotenv()
ors_token = os.getenv("OpenRouteServiceApiKey")
print("Token ok.") if ors_token else print("Token not found.")

def generate_waypoint(pz, distance_km):
    bearing = random.uniform(0, 360)
    destination = geodesic(kilometers=distance_km).destination((pz[1], pz[0]), bearing)
    return [destination.longitude, destination.latitude]

def calculate_wait_time(requests_made_today=0, requests_made_this_minute=0):
    now = datetime.now()
    seconds_in_day = 86400
    seconds_in_minute = 60

    # Calculate remaining requests for the day and minute
    remaining_requests_today = 2000 - requests_made_today
    remaining_requests_this_minute = 40 - requests_made_this_minute

    # Calculate the time left in the day and minute
    time_left_in_day = (datetime.combine(now.date() + timedelta(days=1), datetime.min.time()) - now).total_seconds()
    time_left_in_minute = seconds_in_minute - now.second

    # Calculate wait times
    wait_time_day = time_left_in_day / remaining_requests_today if remaining_requests_today > 0 else seconds_in_day
    wait_time_minute = time_left_in_minute / remaining_requests_this_minute if remaining_requests_this_minute > 0 else seconds_in_minute

    # Return the maximum wait time needed
    return max(wait_time_day, wait_time_minute)

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': ors_token,
    'Content-Type': 'application/json; charset=utf-8'
}

df_locs = pd.read_csv(locs_file)
wait_time = calculate_wait_time()
print(f"Wait time: {wait_time} seconds.")

for index, row in df_locs[:3].iterrows():
    for c in range(0,4):
        pz = [row['longitude'],row['latitude']]

        dist1 = random.uniform(5, 50)
        dist2 = random.uniform(5, 50)

        wp1 = generate_waypoint(pz, dist1)
        wp2 = generate_waypoint(pz, dist2)
        body = {"coordinates":[pz, wp1, wp2, pz]}
        call = requests.post(f'https://api.openrouteservice.org/v2/directions/{profiles[2]}/gpx', json=body, headers=headers)

        if call.status_code == 200:
            gpx_file_path = os.path.join(output_filedir, f'route_loc{str(index).zfill(3)}_{str(c)}.gpx')
            with open(gpx_file_path, 'w') as file:
                file.write(call.text)
        else:
            print(f"Loc. {index} failed. Error code: {call.status_code}\n   --> {call.text}")
        sleep(wait_time)

Token ok.
Wait time: 13.2575515365 seconds.
Loc. 0 failed. Error code: 404
   --> {"error":{"code":2010,"message":"Could not find routable point within a radius of 350.0 meters of specified coordinate 1: -82.8897118 40.9320617."},"info":{"engine":{"build_date":"2024-12-02T11:09:21Z","graph_version":"1","version":"9.0.0"},"timestamp":1734104285082}}
Loc. 1 failed. Error code: 404
   --> {"error":{"code":2010,"message":"Could not find routable point within a radius of 350.0 meters of specified coordinate 2: -82.2344502 41.0917932."},"info":{"engine":{"build_date":"2024-12-02T11:09:21Z","graph_version":"1","version":"9.0.0"},"timestamp":1734104298521}}
Loc. 2 failed. Error code: 404
   --> {"error":{"code":2010,"message":"Could not find routable point within a radius of 350.0 meters of specified coordinate 2: -82.2007483 40.6438499."},"info":{"engine":{"build_date":"2024-12-02T11:09:21Z","graph_version":"1","version":"9.0.0"},"timestamp":1734104311916}}
